In [ ]:
!pip install transformers torch

In [6]:
from transformers import pipeline
import re

# Contador para las métricas de Álvaro Peña (CFO)
consultas_resueltas = 0

print("Cargando cerebro de SegurPlus...")
clasificador = pipeline("zero-shot-classification", model="Recognai/bert-base-spanish-wwm-cased-xnli")

etiquetas_candidatas = [
    "accidente de coche o daños en vehículo",
    "duda sobre facturas y pagos",
    "consultar coberturas de la póliza",
    "proporcionar datos personales como DNI, teléfono o cuenta",
    "poner una queja o reclamación"
]

print("-" * 50)
print("BIENVENIDA (Aviso DPO): Este chat es monitorizado por seguridad.")
print("Por favor, no compartas datos personales sensibles.")
print("-" * 50)
print("¡Chatbot SegurPlus v8 (Versión Final de Negocio) listo!")

while True:
    texto_usuario = input("\nTú: ")
    if texto_usuario.strip().lower() in ['salir', 'exit', 'quit']:
        # Al salir, mostramos métricas para Marta Ruiz
        print(f"\n--- INFORME PARA DIRECCIÓN ---")
        print(f"Consultas que han evitado llamada al call center: {consultas_resueltas}")
        break

    if not texto_usuario.strip(): continue

    # Bloqueo DNI (RGPD - Javier Molina)
    if re.search(r'\d{8}[A-Z]', texto_usuario.upper()):
        intent_detectado = "proporcionar datos personales como DNI, teléfono o cuenta"
        confianza = 1.0
    else:
        resultado = clasificador(texto_usuario, candidate_labels=etiquetas_candidatas)
        intent_detectado = resultado['labels'][0]
        confianza = resultado['scores'][0]

    # RESPUESTAS SEGÚN STAKEHOLDERS
    if intent_detectado == "proporcionar datos personales como DNI, teléfono o cuenta":
        respuesta = "⚠️ [BLOQUEO SEGURIDAD]: Por RGPD no puedo leer DNIs. Usa solo tu número de póliza."

    elif intent_detectado == "consultar coberturas de la póliza":
        # Lucía Torres: Derivación con resumen
        respuesta = "Derivando a humano. MOTIVO: Consulta de coberturas técnicas. El agente recibirá tu historial."

    elif intent_detectado == "poner una queja o reclamación":
        print("Chatbot: Siento tu mala experiencia.")
        detalle_queja = input("Describe el motivo para el expediente: ")
        with open("quejas_registradas.txt", "a") as f:
            f.write(f"QUEJA: {detalle_queja}\n----------------\n")
        respuesta = "Queja registrada (#8823). Esto ahorra tiempo de gestión manual."
        consultas_resueltas += 1 # Sumamos éxito para el CFO

    elif intent_detectado == "accidente de coche o daños en vehículo":
        respuesta = "Entendido. Para abrir el parte necesito matrícula o póliza. (Evitando llamada al 902)."
        consultas_resueltas += 1

    else:
        respuesta = "Entiendo tu consulta. ¿Deseas que lo gestione yo o prefieres un agente?"
        consultas_resueltas += 1

    print(f"Chatbot: {respuesta}")
    print(f"  [Log Auditoría -> Intent: '{intent_detectado}' | Confianza: {confianza:.2f}]")

Cargando cerebro de SegurPlus...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: Recognai/bert-base-spanish-wwm-cased-xnli
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--------------------------------------------------
BIENVENIDA (Aviso DPO): Este chat es monitorizado por seguridad.
Por favor, no compartas datos personales sensibles.
--------------------------------------------------
¡Chatbot SegurPlus v8 (Versión Final de Negocio) listo!

Tú: Mi DNI es 12345678Z
Chatbot: ⚠️ [BLOQUEO SEGURIDAD]: Por RGPD no puedo leer DNIs. Usa solo tu número de póliza.
  [Log Auditoría -> Intent: 'proporcionar datos personales como DNI, teléfono o cuenta' | Confianza: 1.00]

Tú: ¿Me cubre el seguro si se me inunda la cocina?
Chatbot: Derivando a humano. MOTIVO: Consulta de coberturas técnicas. El agente recibirá tu historial.
  [Log Auditoría -> Intent: 'consultar coberturas de la póliza' | Confianza: 0.68]

Tú: Quiero poner una reclamación
Chatbot: Siento tu mala experiencia.
Describe el motivo para el expediente: He tenido un golpe con el coche
Chatbot: Queja registrada (#8823). Esto ahorra tiempo de gestión manual.
  [Log Auditoría -> Intent: 'poner una queja o r